In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [4]:
df=pd.read_csv("output.csv")
df.head(10)

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
5,Now I am getting angry and I want my damn pho.,0
6,Honeslty it didn't taste THAT fresh.),0
7,The potatoes were like rubber and you could te...,0
8,The fries were great too.,1
9,A great touch.,1


In [7]:
df.value_counts().sum()

np.int64(1000)

In [8]:
df.isnull().sum()

 Review    0
Liked      0
dtype: int64

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0    Review  1000 non-null   object
 1   Liked    1000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


In [14]:
df["review_length"] = df[" Review"].apply(lambda x: len(str(x).split()))
df["review_length"]

0       4
1       4
2       8
3      15
4      12
       ..
995    12
996     3
997    10
998    16
999    28
Name: review_length, Length: 1000, dtype: int64

# TASK 2: Clean & Preprocess the Text


In [17]:
df["review"] = df[" Review"].str.lower()

In [19]:
import re

# Remove punctuation from dataframe column
df["review"] = df[" Review"].apply(
    lambda x: re.sub(r'[^\w\s]', '', str(x))
)

In [20]:
df["review"] = df[" Review"].apply(
    lambda x: re.sub(r'\d+', '', str(x))
)

In [22]:
df["review"] = df[" Review"].apply(
    lambda x: re.sub(r'\s+', ' ', str(x)).strip()
)

In [23]:
df["review"]

0                               Wow... Loved this place.
1                                     Crust is not good.
2              Not tasty and the texture was just nasty.
3      Stopped by during the late May bank holiday of...
4      The selection on the menu was great and so wer...
                             ...                        
995    I think food should have flavor and texture an...
996                             Appetite instantly gone.
997    Overall I was not impressed and would not go b...
998    The whole experience was underwhelming, and I ...
999    Then, as if I hadn't wasted enough of my life ...
Name: review, Length: 1000, dtype: object

## Remove stopwords 

In [26]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [27]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [29]:
df["review"] = df[" Review"].apply(
    lambda x: ' '.join(
        word for word in str(x).split()
        if word.lower() not in stop_words
    )
)

In [30]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

df["review"] = df[" Review"].apply(
    lambda x: ' '.join(
        stemmer.stem(word) for word in str(x).split()
    )
)

In [31]:
import re

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        "]+",
        flags=re.UNICODE
    )
    
    return emoji_pattern.sub(r'', str(text))

df["review"] = df[" Review"].apply(remove_emojis)

# TASK 3: Convert Text to Numerical Features


In [32]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

x = cv.fit_transform(df[" Review"])

In [33]:
bow_df = pd.DataFrame(
    x.toarray(),
    columns=cv.get_feature_names_out()
)

bow_df

,00,10,100,11,12,15,17,1979,20,2007,...,yelpers,yet,you,your,yourself,yucky,yukon,yum,yummy,zero
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# TASK 4: Split the Data


In [36]:
from sklearn.model_selection import train_test_split

# Features and target
X = df[" Review"]
y = df["Liked"]

# Split data
x_train, x_test, y_train, y_test = train_test_split(X,y,test_size=0.2)


print(x_train.shape)
print(x_test.shape)

(800,)
(200,)


# TASK 5: Train Naïve Bayes Models


In [38]:
x_train_bow = cv.fit_transform(x_train)
x_test_bow = cv.transform(x_test)

# Train Bernoulli Naive Bayes model
bnb = BernoulliNB()

bnb.fit(x_train_bow, y_train)


,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [40]:
y_pred = bnb.predict(x_test_bow)
y_pred

array([0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0,
       1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       0, 0])

In [42]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))

# Classification report
print(classification_report(y_test, y_pred))

Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.87      0.74      0.80       108
           1       0.74      0.87      0.80        92

    accuracy                           0.80       200
   macro avg       0.81      0.81      0.80       200
weighted avg       0.81      0.80      0.80       200



In [44]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Model
bnb = BernoulliNB()

# Train
bnb.fit(x_train_bow, y_train)

# Predict
y_pred_bnb = bnb.predict(x_test_bow)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred_bnb))
print("Precision:", precision_score(y_test, y_pred_bnb))
print("Recall   :", recall_score(y_test, y_pred_bnb))
print("F1 Score :", f1_score(y_test, y_pred_bnb))

# Confusion Matrix
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_bnb))
# Classification Report
print("\nClassification Report")
print(classification_report(y_test, y_pred_bnb))

Accuracy : 0.8
Precision: 0.7407407407407407
Recall   : 0.8695652173913043
F1 Score : 0.8

Confusion Matrix
[[80 28]
 [12 80]]

Classification Report
              precision    recall  f1-score   support

           0       0.87      0.74      0.80       108
           1       0.74      0.87      0.80        92

    accuracy                           0.80       200
   macro avg       0.81      0.81      0.80       200
weighted avg       0.81      0.80      0.80       200



## TASK 7: Predict Sentiment of New Reviews


In [46]:
sample_reviews = [
    "The food was fantastic!",
    "Worst service ever."
]

# Convert text into vectors
sample_vectors = cv.transform(sample_reviews)

# Predict sentiment
predictions = bnb.predict(sample_vectors)

# Show results
for review, pred in zip(sample_reviews, predictions):
    print("Review :", review)
    print("Prediction :", pred)
    print("-" * 40)

Review : The food was fantastic!
Prediction : 1
----------------------------------------
Review : Worst service ever.
Prediction : 0
----------------------------------------


In [47]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [52]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Load dataset
df = pd.read_csv("output.csv")

# Features and target
X = df[" Review"]
y = df["Liked"]

# Split
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Vectorization
cv = CountVectorizer()

x_train_bow = cv.fit_transform(x_train)

# Train model
model = MultinomialNB()

model.fit(x_train_bow, y_train)

# Save model
joblib.dump(model, "sentiment_model.pkl")

# Save vectorizer
joblib.dump(cv, "vectorizer.pkl")

print("Files saved successfully")

Files saved successfully
